# 🤖 Workshop Day 2: Building Agents — Building Agents with Google ADK

**Agentic RAG Workshop** | Day 2 of 3

---

### 🎯 Today We Will Learn
1. **What is an Agent** — how is it different from a Chatbot?
2. **Google ADK** — Google's tool for building Agents
3. **Tool** — teaching an Agent to "do work"
4. **RAG Tool** — connecting an Agent to the VectorDB built on Day 1
5. **Multi-Agent** — multiple Agents working together
6. **Full Agentic RAG** — search + summarize + remember context

### 🗺️ Pipeline Overview

| Day 1 (Data Engineering) | ➡️ | Day 2 (Agent Building) |
|---|---|---|
| Raw Data → Dedup → Chunk | | 2.1 First Agent |
| → Embed | | 2.2 Add Tools |
| → Index (Qdrant) | ➡️ | 2.3 RAG Tool + Qdrant ⭐ |
| → Retrieve | | 2.4 Multi-Agent |
| | | 2.5 Session & Memory |
| | | 2.6 Full Agentic RAG ⭐ |

## 📦 Section 0: Install Dependencies

> ⚠️ Google ADK is normally used on the terminal with the command `adk web`, but we will adapt it to work on Colab

### 🆚 ADK vs LangChain vs CrewAI

| | **Google ADK** | **LangChain** | **CrewAI** |
|---|:---:|:---:|:---:|
| **Best for** | Multi-Agent + Gemini | LLM+Tool chains | Agent teams |
| **LLM** | Gemini (main) + others | All LLMs | All LLMs |
| **Multi-Agent** | ✅ Built-in | ❌ Must build yourself | ✅ Built-in |
| **Workflow** | Sequential/Parallel/Loop | LCEL Chain | Sequential/Hierarchical |
| **Dev UI** | ✅ `adk web` | ❌ | ❌ |
| **Developed by** | Google | LangChain Inc. | CrewAI Inc. |

> 💡 We choose **ADK** because it is Google's latest framework, designed specifically for Multi-Agent systems
> and it works best with **Gemini**

In [ ]:
%%time
# Install Google ADK and libraries from Day 1
!pip install -q google-adk \
    google-genai \
    sentence-transformers \
    qdrant-client \
    langchain-text-splitters

print('✅ Installation complete!')

In [ ]:
%%time
# Set Gemini API Key
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets')
except Exception:
    api_key = input('🔑 Please paste your Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

# Test API
from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='Reply only with Hello')
    print(f'✅ API works: {(resp.text.strip() if resp.text else '(no output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')
    print('💡 Check: 1) Is the API Key correct? 2) Is there internet? 3) Has the free tier quota run out?')

### 📚 Understanding Gemini API Parameters

When calling Gemini (either directly or through ADK), many parameters can be adjusted:

#### 🎛️ Generation Parameters

| Parameter | Default Value | Range | Purpose |
|-----------|:-----------:|------|----------|
| `temperature` | 1.0 | 0.0 - 2.0 | The "creativity" of the response |
| `top_p` | 0.95 | 0.0 - 1.0 | Limits the pool of candidate words (nucleus sampling) |
| `top_k` | 40 | 1 - ∞ | Number of words considered at each step |
| `max_output_tokens` | 8192 | 1 - 65536 | Limits response length |

#### 🌡️ Temperature — Adjust Creativity

```
temperature = 0.0  →  Gives the same answer every time (deterministic)
                       Best for: grading, factual summaries, RAG

temperature = 1.0  →  Balanced between creativity and correctness
                       Best for: general chatbots, Agents

temperature = 2.0  →  Highly creative, may "go off track" sometimes
                       Best for: brainstorming, storytelling
```

#### 🔍 top_p vs top_k

| | top_k | top_p |
|---|------|------|
| **How words are selected** | Choose from the top k most likely words | Choose from words whose total probability ≤ p |
| **Example** | top_k=3 → choose from 3 words | top_p=0.9 → choose from words covering 90% probability |
| **Effect when lowered** | Lower → more conservative | Lower → more conservative |

#### 🏷️ Available Gemini Models

| Model | Speed | Price | Best for |
|-------|:--------:|:----:|----------|
| `gemini-2.5-flash` | ⚡ Fast | 💰 Cheap | Agent, Tool calling, RAG |
| `gemini-2.5-pro` | 🐢 Slower | 💰💰 Expensive | Deep analysis, grading |
| `gemini-3.1-flash` | ⚡⚡ Fastest | 💰 Cheap | New-generation Agents, general tasks |

> 💡 In this workshop, we use **`gemini-2.5-flash`** because it is fast, affordable, and supports Tool calling well

In [ ]:
%%time
# ─── Try adjusting Temperature ───
from google import genai

prompt = 'Write 1 sentence about AI'

print('🧪 Testing Temperature (same prompt 3 times)\n')

for temp in [0.0, 1.0, 2.0]:
    print(f'--- temperature = {temp} ---')
    for round_num in range(1, 4):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    temperature=temp,
                    max_output_tokens=100,
                )
            )
            print(f'  Round {round_num}: {(resp.text.strip() if resp.text else '(no output)')[:80]}')
        except Exception as e:
            print(f'  ❌ Error: {e}')
    print()

print('💡 Notice: temp=0.0 → same answer every round, temp=2.0 → different answers')

In [ ]:
%%time
# ─── Try adjusting max_output_tokens ───
print('🧪 Testing max_output_tokens\n')

for max_tokens in [20, 100, 500]:
    try:
        resp = client.models.generate_content(
            model='gemini-2.5-flash',
            contents='Explain what AI is',
            config=genai.types.GenerateContentConfig(
                max_output_tokens=max_tokens,
                temperature=0.5
            )
        )
        text = resp.text.strip() if resp.text else '(no output — too few tokens)'
        print(f'max_tokens={max_tokens:>3}: ({len(text)} chars) {text[:80]}...')
    except Exception as e:
        print(f'  ❌ Error: {e}')

print('\n💡 max_output_tokens limits "length", not "correctness"')

### 💡 Observations
- `temperature=0.0` → answers are **the same** every round — good for RAG where consistent answers are needed
- `temperature=2.0` → answers are **different** every round — good for creative work
- Low `max_output_tokens` → the answer gets cut off, not "summarized shorter"

> 🎯 For **Agent/RAG**, recommended: `temperature=0.5-1.0`, `max_output_tokens=2048+`

---
## 🤖 Section 2.1: Your First Agent

### Agent vs Chatbot — What's the difference?

| | Chatbot | Agent |
|---|---------|-------|
| **How it works** | Only answers questions | Makes decisions + calls tools |
| **Capabilities** | Answers from what it knows | Searches information, calculates, calls APIs |
| **Memory** | Usually does not remember | Can remember context |
| **Examples** | FAQ Bot | ChatGPT + Plugins, Gemini |

### What is Google ADK?

**Agent Development Kit (ADK)** = an open-source Python framework from Google

```
Agent = Model (LLM) + Instructions (how to think) + Tools
```

- Uses **Gemini** as the main LLM (supports other models too)
- Makes it easy to build **Multi-Agent** systems
- Has a **Web UI** for debugging

In [ ]:
%%time
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

import warnings
warnings.filterwarnings('ignore')

# ─── First Agent: no Tools, only Instructions ───
greeting_agent = Agent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='An Agent that greets and answers questions in Thai',
    instruction="""You are a Thai AI Assistant
    - Greet politely
    - Answer briefly, no more than 2 sentences
    - Use emojis appropriately
    """
)

print('✅ First Agent created successfully!')
print(f'   Name: {greeting_agent.name}')
print(f'   Model: {greeting_agent.model}')

### 🔧 Why use Async + Runner?

- **`async/await`** — ADK is designed to work asynchronously because the Agent waits for the LLM response (I/O bound)
- **`InMemoryRunner`** — runs the Agent in memory and manages the event loop
- **`InMemorySessionService`** — stores conversation history in RAM

```
Runner → create Session → send message to Agent → Agent makes decisions
  → calls Tool (if needed) → generates response → sends Event back
```

> 💡 On Colab, you can use `await` directly in a cell (no need for `asyncio.run()`)

In [ ]:
# ─── Helper function to chat with an Agent on Colab ───
import asyncio

async def chat_with_agent(runner, message, user_id='user_1', session_id='default_session'):
    """Send a message to the Agent and receive the response"""
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=message)]
    )
    
    final_response = ''
    tool_calls = []
    
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_response = part.text
                if part.function_call:
                    tool_calls.append({
                        'name': part.function_call.name,
                        'args': dict(part.function_call.args) if part.function_call.args else {}
                    })
    
    if tool_calls:
        print(f'  🔧 Tools called: {[t["name"] for t in tool_calls]}')
    
    return final_response


async def demo_chat(agent, messages):
    """Demo multi-turn conversation with an Agent (using the same session throughout)"""
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f'session_{id(agent)}'
    
    # Create session through runner
    await runner.session_service.create_session(
        app_name=agent.name,
        user_id='user_1',
        session_id=session_id
    )
    
    for msg in messages:
        print(f'\n👤 User: {msg}')
        response = await chat_with_agent(runner, msg, session_id=session_id)
        print(f'🤖 Agent: {response}')


print('✅ Chat function is ready to use')

In [ ]:
# ─── Test the first Agent ───
await demo_chat(greeting_agent, [
    'Hello',
    'What is AI? Explain briefly',
    'Thank you'
])

### 💡 Observations
- The Agent answers **according to the instruction** we defined (polite, short, with emoji)
- If you change the instruction → the Agent's personality changes immediately
- The Agent still has no Tool → it can only answer from the **LLM's knowledge**

> 🎯 **Instruction is the soul of the Agent** — write it well = smarter Agent

### 🧪 Exercise 2.1

Try changing the Agent's `instruction` to give it different personalities:
- An AI teacher
- A doctor giving health advice
- A financial advisor

In [ ]:
# 🧪 Exercise: change the instruction to give the Agent a new personality
# my_agent = Agent(
#     name='...',
#     model='gemini-2.5-flash',
#     description='...',
#     instruction='...'
# )
# await demo_chat(my_agent, ['Hello', 'Can you give me some advice?'])

### ✍️ Tips: Writing Good Instructions

**Instruction = the brain of the Agent** — write it well and you get good results; write it broadly and results become unpredictable

| Technique | Example |
|--------|--------|
| **Specify the Role** | "You are a health expert" |
| **Say when to use a Tool** | "Use calculate_bmi when asked about weight" |
| **Say when to answer directly** | "General questions such as greetings → answer directly" |
| **Define the Format** | "Answer in bullet points, no more than 5 items" |
| **Add Constraints** | "Do not guess; if you don't know, say so clearly" |
| **Language** | "Answer in Thai and use emoji" |

```python
# ❌ Bad instruction
instruction = "Answer questions"

# ✅ Good instruction
instruction = """You are a health expert. Answer in Thai.
- Use calculate_bmi when asked about BMI or weight
- General questions → answer directly without calling a tool
- Keep the answer concise, no more than 3 sentences
- Do not guess; if unsure, recommend seeing a doctor"""
```

---
## 🔧 Section 2.2: Create a Tool for the Agent

### What is a Tool?

**Tool = a Python function that the Agent can call**

```
User asks: "What is my BMI? Weight 70 kg, height 175 cm"
    ↓
Agent reads the question → decides: must use the calculate_bmi tool!
    ↓
Agent calls: calculate_bmi(weight_kg=70, height_cm=175)
    ↓
Agent gets the result → generates an answer for the user
```

### 📌 Important Rules for Writing Tools

| Rule | Why? |
|-----|-------|
| Must have a **docstring** | The LLM reads the docstring to choose the tool |
| Must have **type hints** | The LLM knows what parameters to send |
| Return a **dict** | The Agent reads results from the dict |
| Function name should be clear | The LLM also chooses tools by name |

In [ ]:
%%time
# ─── Tool 1: Calculate BMI ───
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """Calculate Body Mass Index (BMI) from weight and height

    Args:
        weight_kg: weight in kilograms
        height_cm: height in centimeters

    Returns:
        dict: BMI value, weight category, and advice
    """
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    if bmi < 18.5:
        level = 'Underweight'
        advice = 'You should gain weight with a high-protein diet'
    elif bmi < 25:
        level = 'Normal weight'
        advice = 'Maintain this weight and exercise regularly'
    elif bmi < 30:
        level = 'Overweight'
        advice = 'You should lose weight through diet control and exercise'
    else:
        level = 'Obese'
        advice = 'You should consult a doctor to plan weight loss'

    return {'bmi': round(bmi, 1), 'level': level, 'advice': advice}


# ─── Tool 2: Convert temperature ───
def convert_temperature(value: float, from_unit: str) -> dict:
    """Convert temperature between Celsius and Fahrenheit

    Args:
        value: temperature value
        from_unit: source unit ('celsius' or 'fahrenheit')

    Returns:
        dict: converted temperature value
    """
    if from_unit.lower() == 'celsius':
        result = (value * 9/5) + 32
        return {'result': round(result, 1), 'from': f'{value}°C', 'to': f'{result:.1f}°F'}
    elif from_unit.lower() == 'fahrenheit':
        result = (value - 32) * 5/9
        return {'result': round(result, 1), 'from': f'{value}°F', 'to': f'{result:.1f}°C'}
    else:
        return {'error': f'Unknown unit: {from_unit}'}


# ─── Create Agent with Tools ───
health_agent = Agent(
    name='health_agent',
    model='gemini-2.5-flash',
    description='A health Agent that calculates BMI and converts temperature',
    instruction="""You are a health Agent. Answer in Thai.
    - Use calculate_bmi when asked about BMI or weight
    - Use convert_temperature when asked about temperature
    - If the question is unrelated to the tools, answer from general knowledge
    """,
    tools=[calculate_bmi, convert_temperature]
)

print('✅ Created Health Agent with 2 tools')
print(f'   🔧 Tools: {[t.__name__ for t in health_agent.tools]}')

In [ ]:
runner = InMemoryRunner(agent=health_agent, app_name=health_agent.name)
await runner.session_service.create_session(app_name=health_agent.name, user_id='user_1', session_id='test_session')
# ─── Test Tool Selection ───
print('═' * 60)
print('🧪 Test: Which Tool does the Agent choose?')
print('═' * 60)

test_messages = [
    'Weight 70 kg, height 175 cm, what is my BMI?',
    '37 degrees Celsius equals how many Fahrenheit?',
    'What is AI?',  # no tool needed!
]

for msg in test_messages:
    print(f'\n👤 User: {msg}')
    response = await chat_with_agent(runner, msg, session_id='test_session')
    print(f'🤖 Agent: {response}')

### 💡 Observations
- BMI question → Agent calls `calculate_bmi` 🔧
- Temperature question → Agent calls `convert_temperature` 🔧
- General question → Agent **does not call a tool** and answers directly 💬

> The Agent **decides by itself** which tool to use — this is what makes it different from a chatbot!

### 🧪 Exercise 2.2

Create 1 tool of your own and add it to the Agent:
- Calculate grade from score (score → grade)
- Calculate interest
- Convert currency
- Anything else you can think of!

In [ ]:
# 🧪 Exercise: create your own tool
# def my_tool(param1: float, param2: str) -> dict:
#     """Describe the tool (important! The LLM reads this)"""
#     ...
#     return {...}
#
# my_agent = Agent(
#     name='my_agent', model='gemini-2.5-flash',
#     tools=[my_tool, calculate_bmi]
# )
# await demo_chat(my_agent, ['Test the new tool'])

---
## 🔍 Section 2.3: RAG Tool — Connect the Agent to Qdrant ⭐

### What is Agentic RAG?

```
Regular RAG:     Ask → Search → Answer              (searches every time, no matter the question)
Agentic RAG:     Ask → Agent decides → Search? → Answer    (searches only when needed)
```

### Why is Agentic RAG better?

| | Regular RAG | Agentic RAG |
|---|----------|-------------|
| Ask "Hello" | Search VectorDB (wastes time) | Answer directly, no search ✅ |
| Ask "How does AI help banks?" | Search VectorDB ✅ | Search VectorDB ✅ |
| Ask follow-up | Search again every time | Remembers context + searches only when needed ✅ |
| Use multiple tools | ❌ Cannot | ✅ Chooses tools itself |

### Real-World Examples
- **KADE (Kasikornbank)**: Agent decides whether a question is about loans / credit cards / investments, then searches the relevant KB
- **Chula Study Bot**: Agent is asked "What exams do I have?" → searches textbooks, asked "Hello" → greets directly

> The Agent decides for itself **when** it needs to search and **where** to search

In [ ]:
%%time
# ─── Prepare Qdrant VectorDB (same data as Day 1) ───
import hashlib, random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# Load Embedding Model
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print('✅ Loaded embedding model')

# Same data as Day 1
documents = {
    'healthcare': 'Siriraj Hospital uses AI to help analyze chest X-ray images, reducing diagnosis time from 30 minutes to 5 minutes. The Deep Learning system can detect abnormalities with 95% accuracy, helping doctors screen tuberculosis patients faster, reducing waiting time for test results, and increasing the chance of timely treatment.',
    'banking': 'Kasikornbank developed the KADE AI Assistant using RAG to retrieve financial product information and answer customer questions, reducing response time from 5 minutes to 30 seconds. The system understands Thai and English, and can automatically handle questions about loans, credit cards, and investments.',
    'education': 'Chulalongkorn University built a RAG system for question answering from textbooks, converting more than 500 PDF books into vector embeddings. It uses a multilingual model that supports Thai. Students can ask questions and receive answers with page references, reducing search time from hours to just seconds.',
    'agriculture': 'The Department of Agriculture uses AI to analyze satellite and drone images to detect crop diseases in rice fields. It developed a Computer Vision system to identify blast disease and leaf spot disease with 92% accuracy, helping farmers detect problems earlier and reduce crop damage by 40%.',
    'kmitl': 'King Mongkut’s Institute of Technology Ladkrabang (KMITL), Faculty of Engineering, developed AI systems for a Smart Campus. It uses IoT sensors combined with Machine Learning to analyze building energy consumption, reducing electricity costs by 25%. It also developed Thai NLP systems for analyzing student feedback and used RAG to build an AI Tutor that answers course questions from teaching materials.',
    'nlp': 'Natural Language Processing, or NLP, for the Thai language has unique challenges because Thai has no spaces between words. This requires specialized word segmentation tools such as PyThaiNLP. In addition, vowels, tone marks, and Thai numerals add more complexity to tokenization.',
    'rag_architecture': 'The RAG architecture consists of 3 main parts: the Retriever, which searches for documents from the VectorDB; the Reader, which reads and understands the documents; and the Generator, which creates answers from the retrieved information. RAG helps reduce LLM hallucination because answers are grounded in real data.',
    'embedding': 'Text Embedding converts text into numeric vectors. Microsoft’s multilingual-e5-large model supports more than 100 languages, including Thai, and creates 1024-dimensional vectors. Texts with similar meanings will have vectors that are close together in embedding space.',
    'vector_db': 'Qdrant is a high-performance Vector Database that supports ANN (Approximate Nearest Neighbor) search, enabling fast retrieval even with millions of vectors. It supports Cosine Similarity, Dot Product, and Euclidean Distance.',
}

# Chunk + Embed + Index
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

all_chunks = []
all_sources = []
for source, text in documents.items():
    chunks = splitter.split_text(text)
    for chunk in chunks:
        all_chunks.append(chunk)
        all_sources.append(source)

print(f'📊 {len(all_chunks)} chunks from {len(documents)} documents')

# Embed
passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]
print(f'✅ Embedding: {embeddings.shape}')

# Create Qdrant collection
qdrant = QdrantClient(':memory:')
qdrant.create_collection(
    collection_name='thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

# Upsert
points = [
    models.PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={'text': all_chunks[i], 'source': all_sources[i]}
    )
    for i in range(len(all_chunks))
]
qdrant.upsert(collection_name='thai_ai_kb', points=points)
print(f'✅ Upserted {len(points)} points into Qdrant')

In [ ]:
%%time
# ─── Create RAG Tool ───
def search_thai_ai_knowledge(query: str) -> dict:
    """Search for AI-related information in Thailand from the knowledge base

    Use this tool when the user asks about:
    - AI applications in Thailand (healthcare, banking, education, agriculture)
    - RAG, Embedding, Vector Database techniques
    - Thai NLP

    Args:
        query: the question to search for, in Thai or English

    Returns:
        dict: results from the knowledge base
    """
    query_vector = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points(
        collection_name='thai_ai_kb',
        query=query_vector,
        limit=3
    ).points

    if not results:
        return {'status': 'no_results', 'message': 'No relevant information found'}

    contexts = []
    for r in results:
        contexts.append({
            'text': r.payload['text'],
            'source': r.payload['source'],
            'score': round(r.score, 4)
        })

    return {
        'status': 'success',
        'num_results': len(contexts),
        'results': contexts
    }

print('✅ RAG Tool is ready to use')

# Test the tool directly
test = search_thai_ai_knowledge('How does AI help doctors diagnose diseases?')
print(f'\n🧪 Test: {test["num_results"]} results')
for r in test['results']:
    print(f'  [{r["source"]}] score={r["score"]} | {r["text"][:60]}...')

In [ ]:
%%time
# ─── Create RAG Agent ───
rag_agent = Agent(
    name='thai_ai_expert',
    model='gemini-2.5-flash',
    description='AI expert in Thailand, searching from a knowledge base',
    instruction="""You are an expert in AI in Thailand

    Rules:
    1. When the user asks about AI in Thailand, case studies, or RAG techniques
       → always use search_thai_ai_knowledge to search before answering
    2. Answer based on information from the knowledge base and cite the source
    3. If no information is found, say clearly: "There is no information in the knowledge base"
    4. General questions such as greetings → answer directly without searching
    5. Answer in Thai, be concise, and use bullet points
    """,
    tools=[search_thai_ai_knowledge]
)

print('✅ RAG Agent created successfully!')

In [ ]:
# ─── Demo: Agent decides by itself ───
print('═' * 60)
print('🧪 Demo: When does the Agent choose to search, and when does it answer directly?')
print('═' * 60)

await demo_chat(rag_agent, [
    'Hello',                                  # → answers directly
    'How does AI help healthcare in Thailand?', # → searches Qdrant
    'What is the architecture of RAG?',        # → searches Qdrant
    'Thank you',                               # → answers directly
])

### 💡 Observations
- "Hello" → Agent **does not call a tool** and answers directly 💬
- "How does AI help healthcare?" → Agent **calls** `search_thai_ai_knowledge` 🔧
- The answer is **grounded in real data** from Qdrant, not made up

> This is the heart of **Agentic RAG**: the Agent decides → searches when needed → answers from real data

### 🧪 Exercise 2.3

1. Ask the RAG Agent about "Thai NLP" and "Vector Database" and see whether it answers from the knowledge base
2. Try adding new data into Qdrant (for example, logistics or retail) and then ask the Agent

In [ ]:
# 🧪 Exercise: test the RAG Agent
# await demo_chat(rag_agent, [
#     'What problems does Thai NLP have?',
#     'Why is Qdrant good?',
# ])

---
## 👥 Section 2.4: Multi-Agent System

### Why Multi-Agent?

One Agent doing everything → gets confused when things become complex  
Multi-Agent: **divide work by expertise** → more accurate

### 🧩 3 Workflow Patterns in ADK

| Pattern | Description | When to use? |
|---------|--------|-------------|
| **Sequential** | Do one step at a time A→B→C | When information must be passed along, e.g. search→summarize→translate |
| **Parallel** | Do tasks at the same time A‖B‖C | For independent tasks, e.g. search 3 sources at once |
| **Loop** | Repeat until satisfied | For improving answers, e.g. write→review→fix→review... |

```
Sequential:    [A] → [B] → [C]          done in order

Parallel:      [A] ─┐
               [B] ─┤─→ combine results      done at the same time
               [C] ─┘

Loop:          [A] → [B] →╮              repeated until
                ╰─────────╯              it meets the criteria
```

### 1️⃣ SequentialAgent — One Step at a Time

```
Search for information → Summarize concisely → Translate into English
      Step 1                 Step 2                 Step 3
```

Each agent passes its result to the next agent through **shared state**

In [ ]:
%%time
from google.adk.agents import SequentialAgent

# Sub-agent 1: Search for information
step1_search = Agent(
    name='step1_search',
    model='gemini-2.5-flash',
    description='Searches for AI information in Thailand',
    instruction='Search the knowledge base and answer with facts only',
    tools=[search_thai_ai_knowledge]
)

# Sub-agent 2: Summarize
step2_summary = Agent(
    name='step2_summary',
    model='gemini-2.5-flash',
    description='Summarizes information concisely',
    instruction='Take the information from the previous step and summarize it into 3 bullet points in Thai'
)

# Sub-agent 3: Translate
step3_translate = Agent(
    name='step3_translate',
    model='gemini-2.5-flash',
    description='Translates into English',
    instruction='Translate the given information into natural English'
)

# SequentialAgent: Search → Summarize → Translate
sequential_pipeline = SequentialAgent(
    name='search_summarize_translate',
    description='Pipeline: Search → Summarize → Translate',
    sub_agents=[step1_search, step2_summary, step3_translate]
)

print('✅ Created SequentialAgent: Search → Summarize → Translate')
print(f'   Order: {[a.name for a in sequential_pipeline.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=sequential_pipeline, app_name=sequential_pipeline.name)
await runner.session_service.create_session(app_name=sequential_pipeline.name, user_id='user_1', session_id='test_session')
# ─── Test Sequential Pipeline ───
print('═' * 60)
print('🧪 SequentialAgent: Search → Summarize → Translate')
print('═' * 60)

response = await chat_with_agent(runner, 'How does AI help Thai farmers?', session_id='test_session')
print(f'\n📋 Final result (after 3 steps):')
print(response)

### 💡 Observations
- The final output is in **English** because it passed through `step3_translate`
- If you switch the order of `sub_agents` → the result changes!
- **Shared State**: each agent sees the result from the previous agent

### 2️⃣ ParallelAgent — Work Simultaneously

| Input | Task | Output |
|:-----:|------|:------:|
| Question | → Search healthcare | Combined result |
| | → Search banking | |
| | → Search education | |

**Faster than Sequential** because it does 3 tasks at once instead of waiting one by one

In [ ]:
%%time
from google.adk.agents import ParallelAgent

# 3 agents search different topics in parallel
healthcare_searcher = Agent(
    name='healthcare_searcher',
    model='gemini-2.5-flash',
    description='Searches for AI in healthcare',
    instruction='Search for information about AI in Thai medicine and healthcare. Answer briefly in 2-3 sentences',
    tools=[search_thai_ai_knowledge]
)

banking_searcher = Agent(
    name='banking_searcher',
    model='gemini-2.5-flash',
    description='Searches for AI in finance',
    instruction='Search for information about AI in Thai banking and finance. Answer briefly in 2-3 sentences',
    tools=[search_thai_ai_knowledge]
)

education_searcher = Agent(
    name='education_searcher',
    model='gemini-2.5-flash',
    description='Searches for AI in education',
    instruction='Search for information about AI in Thai education. Answer briefly in 2-3 sentences',
    tools=[search_thai_ai_knowledge]
)

# ParallelAgent: search 3 topics at the same time!
parallel_search = ParallelAgent(
    name='parallel_multi_topic_search',
    description='Search 3 topics in parallel',
    sub_agents=[healthcare_searcher, banking_searcher, education_searcher]
)

print('✅ Created ParallelAgent: search 3 topics in parallel')
print(f'   Agents: {[a.name for a in parallel_search.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=parallel_search, app_name=parallel_search.name)
await runner.session_service.create_session(app_name=parallel_search.name, user_id='user_1', session_id='test_session')
# ─── Test Parallel Search ───
print('═' * 60)
print('🧪 ParallelAgent: Search 3 topics in parallel')
print('═' * 60)

response = await chat_with_agent(runner, 'Summarize the overall AI landscape in Thailand', session_id='test_session')
print(f'\n📋 Combined result (searching 3 domains in parallel):')
print(response)

### 💡 Observations
- 3 agents work **at the same time** → theoretically 3x faster than Sequential
- Each agent is **independent** and does not see the others' results
- Best for tasks that **do not need to pass data** between each other

### 3️⃣ LoopAgent — Repeat Until It Meets the Criteria

| Step | Action | Result |
|:----:|--------|--------|
| 1 | Write the answer | draft |
| 2 | Review quality | not good enough? → go back to Step 1 |
| 3 | Meets criteria | ✅ done |

**Used for self-improvement**: Agent writes → Agent reviews → if not good enough, revise and review again

> ⚠️ You must set `max_iterations` to prevent infinite loops!

In [ ]:
%%time
from google.adk.agents import LoopAgent

# Agent 1: Write summary
writer_agent = Agent(
    name='writer_agent',
    model='gemini-2.5-flash',
    description='Writes a content summary',
    instruction="""Write a concise and easy-to-understand summary of the given content
    If there is feedback from the previous round, improve based on it""",
    tools=[search_thai_ai_knowledge]
)

# Agent 2: Review quality (critic)
critic_agent = Agent(
    name='critic_agent',
    model='gemini-2.5-flash',
    description='Reviews answer quality and provides feedback',
    instruction="""Review the given answer:
    - Is the information complete?
    - Is it easy to understand?
    - Does it include numbers or factual references?
    
    If it is not good enough → suggest what needs improvement
    If it is good enough → reply with 'APPROVED: [final summary]'
    """
)

# LoopAgent: write → review → revise → review → ... (max 3 rounds)
loop_refiner = LoopAgent(
    name='refine_loop',
    description='Improve the answer until it meets the criteria',
    sub_agents=[writer_agent, critic_agent],
    max_iterations=3  # prevent infinite loops!
)

print('✅ Created LoopAgent: Write ↔ Review (max 3 rounds)')
print(f'   Max iterations: {loop_refiner.max_iterations}')

In [ ]:
runner = InMemoryRunner(agent=loop_refiner, app_name=loop_refiner.name)
await runner.session_service.create_session(app_name=loop_refiner.name, user_id='user_1', session_id='test_session')
# ─── Test Loop Agent ───
print('═' * 60)
print('🧪 LoopAgent: Write → Review → Revise → Review ...')
print('═' * 60)

response = await chat_with_agent(runner, 'Summarize how RAG works, with a real example from Thailand', session_id='test_session')
print(f'\n📋 Final result (after refinement):')
print(response)

### 💡 Observations
- Writer writes a summary → Critic reviews → if not good enough, Critic gives feedback → Writer revises
- LoopAgent stops when: (1) Critic replies APPROVED or (2) `max_iterations` is reached
- **The more rounds it loops, the better the answer may get** — but it also takes more time

> ⚠️ If you do not set `max_iterations` → it may loop forever!

### 📊 Comparing 3 Patterns + LLM-based Routing

| Pattern | Class | Flow | LLM controls it? | Example |
|--------|-------|------|---------------|----------|
| **Sequential** | `SequentialAgent` | A→B→C | ❌ Deterministic | Search→Summarize→Translate |
| **Parallel** | `ParallelAgent` | A‖B‖C | ❌ Deterministic | Search 3 sources at once |
| **Loop** | `LoopAgent` | A↔B (repeat) | ❌ Deterministic | Write↔Review until good |
| **LLM Routing** | `Agent` + `sub_agents` | LLM chooses | ✅ LLM decides | Orchestrator delegates tasks |

> 💡 **Workflow Agents** (Sequential/Parallel/Loop) are **deterministic** = they give the same kind of flow every time  
> while **LLM Routing** (`Agent` + `sub_agents`) lets the LLM decide = more flexible but results may vary

### 4️⃣ LLM-based Routing — Agent Decides Who Gets the Task

Use `Agent` + `sub_agents` → the **LLM chooses** who should handle the task

In [ ]:
%%time
# ─── LLM-based Routing: Orchestrator ───

# Specialized sub-agents
search_agent = Agent(
    name='search_agent',
    model='gemini-2.5-flash',
    description='Searches information from the Thai AI knowledge base',
    instruction='Search for information and answer only with the facts found',
    tools=[search_thai_ai_knowledge]
)

summarizer_agent = Agent(
    name='summarizer_agent',
    model='gemini-2.5-flash',
    description='Summarizes information concisely and clearly',
    instruction='Summarize into 3-5 bullet points and add emoji'
)

translator_agent = Agent(
    name='translator_agent',
    model='gemini-2.5-flash',
    description='Translates text between Thai and English',
    instruction='Translate naturally, not word-for-word'
)

# Orchestrator: LLM decides by itself
orchestrator = Agent(
    name='thai_ai_orchestrator',
    model='gemini-2.5-flash',
    description='Main Agent that manages the team',
    instruction="""You are the main Agent with a team:
    1. search_agent — searches for information
    2. summarizer_agent — summarizes
    3. translator_agent — translates
    Delegate tasks to the team appropriately""",
    sub_agents=[search_agent, summarizer_agent, translator_agent]
)

print('✅ Created LLM Routing Orchestrator')

In [ ]:
# ─── Test LLM Routing ───
print('═' * 60)
print('🧪 LLM Routing: Agent delegates tasks to the team')
print('═' * 60)

await demo_chat(orchestrator, [
    'How is AI used in Thai agriculture?',
    'Please summarize it briefly',
    'Translate it into English',
])

### 💡 Observations
- The Orchestrator is **not hard-coded** to send tasks to anyone → the **LLM chooses** based on descriptions
- Ask "search..." → delegate to 
- Ask "summarize..." → delegate to 
- Ask "translate..." → delegate to 

> ⚡ **Advantage**: very flexible, easy to add new agents  
> ⚠️ **Disadvantage**: the LLM may send the task to the wrong agent, and results may differ each time

### 🧪 Exercise 2.4

1. Try creating a new `SequentialAgent`: Search → Analyze → Create quiz
2. Try using `ParallelAgent` to compare AI in Thailand vs England vs Japan
3. Try a `LoopAgent` where the agent writes a poem and reviews it until it becomes good

In [ ]:
# 🧪 Exercise: choose a pattern you like and try building one yourself
# Hint: SequentialAgent, ParallelAgent, LoopAgent
# from google.adk.agents import SequentialAgent  # etc.


---
## 🧠 Section 2.5: Session & Memory

### Agent Can Remember — Through Session

```
Without Session:     every message starts fresh (forgets everything)
With Session:        remembers previous conversation → can respond continuously
```

### What does a Session contain?

| Component | What it is | Example |
|-----------|--------|--------|
| **History** | user ↔ agent message history | "Ask: What is AI?" → "Answer: AI is..." |
| **State** | data stored between turns | user name, preferences, tool results |
| **user_id** | identifies the user | "student_001" |
| **session_id** | identifies the conversation | "session_abc123" |

### What types of SessionService are there?

| Type | Stored in | After program closes | Used when |
|------|--------|-----------|--------|
| `InMemorySessionService` | RAM | ❌ Everything disappears | Dev/testing |
| `DatabaseSessionService` | DB | ✅ Still persists | Production |

> 💡 In this workshop, we use `InMemorySessionService` because it is simple and fast

In [ ]:
# ─── Demo: Agent can remember context ───
async def conversation_with_memory(agent, messages):
    """Multi-turn conversation using the same session → Agent remembers"""
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    
    await runner.session_service.create_session(
        app_name=agent.name,
        user_id='student_demo',
        session_id='memory_demo'
    )

    for msg in messages:
        print(f'\n👤 User: {msg}')
        content = types.Content(
            role='user',
            parts=[types.Part.from_text(text=msg)]
        )

        final_response = ''
        async for event in runner.run_async(
            user_id='student_demo',
            session_id='memory_demo',
            new_message=content
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        final_response = part.text
        
        print(f'🤖 Agent: {final_response}')

# Test
await conversation_with_memory(greeting_agent, [
    'Hello, my name is Somchai',
    'I am interested in AI in healthcare',
    'What about finance?',
    'Please compare these two industries',
    'Do you remember what my name is?',
])

### 💡 Observations
- Ask "What about finance?" → the Agent **remembers** that the conversation is about AI in Thailand
- Ask "Compare these two" → the Agent **remembers** that we discussed healthcare + banking
- If you use `chat_with_agent` (new session every time) → the Agent forgets everything

> **Session = memory**: use the same session → the Agent can remember context

---
## 🚀 Section 2.6: Full Agentic RAG ⭐

Putting everything together: **Agent + RAG Tool + Multi-Agent + Memory**

| Component | Role |
|---|---|
| 🎯 **Study Assistant** | Orchestrator Agent |
| 🔍 Search Agent (Qdrant) | Searches for information |
| 📝 Summarizer Agent | Summarizes information |
| 🌐 Translator Agent | Translates language |
| 💾 Session | Remembers conversation |

In [ ]:
# ─── Create Full Study Assistant ───

study_assistant = Agent(
    name='study_assistant',
    model='gemini-2.5-flash',
    description='AI/RAG study assistant for Thai students',
    instruction="""You are an AI/RAG study assistant for Thai students

    Capabilities:
    1. search_agent — search for information from the knowledge base (use when asked about AI/RAG in Thailand)
    2. summarizer_agent — summarize information clearly
    3. translator_agent — translate language

    How to work:
    - When asked about AI → search using search_agent first → then summarize clearly
    - Remember conversation context. If the user asks "what about..." refer to the previous context
    - Answer in Thai and use emoji
    - At the end of every answer, add 💡 with an extra interesting suggestion
    """,
    sub_agents=[
        Agent(
            name='study_search_agent',
            model='gemini-2.5-flash',
            description='Searches information from the Thai AI knowledge base',
            instruction='Search for information and answer only with the facts found',
            tools=[search_thai_ai_knowledge]
        ),
        Agent(
            name='study_summarizer_agent',
            model='gemini-2.5-flash',
            description='Summarizes information concisely and clearly',
            instruction='Summarize into 3-5 bullet points and add emoji'
        ),
        Agent(
            name='study_translator_agent',
            model='gemini-2.5-flash',
            description='Translates between Thai and English',
            instruction='Translate accurately according to context'
        ),
    ]
)

print('✅ Created full Study Assistant!')

In [ ]:
# ─── Demo: Full Agentic RAG ───
print('═' * 60)
print('🎓 Demo: Study Assistant (Full Agentic RAG)')
print('═' * 60)

await conversation_with_memory(study_assistant, [
    'Please explain how RAG works',
    'Are there real use cases in Thailand?',
    'Please summarize everything',
    'Translate the summary into English',
])

### 🧪 Exercise 2.6

1. Try asking the Study Assistant with your own questions
2. Observe which sub-agent the Agent delegates the task to
3. Try adding a new tool, such as quiz_generator

In [ ]:
# 🧪 Exercise: chat with the Study Assistant
# await conversation_with_memory(study_assistant, [
#     'Which model should be used for Thai embeddings?',
#     'Why do we need multilingual?',
#     'Please summarize briefly',
# ])

---
## 🎯 Summary: What We Learned Today

| Section | What We Learned | Tools |
|---------|------------|----------|
| 2.1 | Agent = LLM + Instructions | `google.adk.agents.Agent` |
| 2.2 | Tool = Python function that the Agent can call | `tools=[function]` |
| 2.3 | RAG Tool = Search VectorDB | Qdrant + Embedding |
| 2.4 | Multi-Agent = Multiple Agents work together | `sub_agents=[]` |
| 2.5 | Session = Agent remembers context | `InMemorySessionService` |
| 2.6 | Agentic RAG = Full system | Everything combined |

### 🛣️ Day 3: Evaluation
- Measure RAG quality with RAGAS
- Automated Testing
- Improve the Agent further

---

**🙏 Thank you for learning together!**